[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github.com/MLinApp-polito/mla-prj-23-project-am04_group-am01/blob/main/defect_detection.ipynb)

TODO: fix the button

# PBF Defect Detection

This notebook covers data loading, training, validation, and inference for detecting defects in Powder Bed Fusion images using a fine-tuned CNN.

## Clone GithHub repo

In [ ]:
!rm -rf mla_project/

In [1]:
import os

if not os.path.exists("/content/mla-prj-23-project-am04_group-am01") and not os.path.exists("/content/mla_project"):
  # DON'T SHARE THE PERSONAL ACCESS TOKEN

  # change the name of the branch here as needed
  !git clone -b diffusion https://***REMOVED-GITHUB-TOKEN***@github.com/MLinApp-polito/mla-prj-23-project-am04_group-am01.git

  # Rename folder for simplicity
  !mv /content/mla-prj-23-project-am04_group-am01 /content/mla_project

!cd /content/mla_project && git pull

Cloning into 'mla-prj-23-project-am04_group-am01'...
remote: Enumerating objects: 2119, done.
remote: Counting objects: 100% (39/39), done.
remote: Compressing objects: 100% (22/22), done.
remote: Total 2119 (delta 18), reused 26 (delta 14), pack-reused 2080 (from 2)
Receiving objects: 100% (2119/2119), 1.75 GiB | 17.35 MiB/s, done.
Resolving deltas: 100% (1029/1029), done.
Updating files: 100% (388/388), done.
Already up to date.


## Install Dependencies

In [ ]:
!pip install torch torchvision matplotlib tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 104.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 91.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 89.8 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitli

## Imports

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

## Dataset mean and std

In [ ]:
!python /content/mla_project/src/data_loader.py --data-dir /content/mla_project/images --compute-stats

Images shape:  torch.Size([16, 1, 1024, 1280])
Images shape:  torch.Size([16, 1, 1024, 1280])
Images shape:  torch.Size([16, 1, 1024, 1280])
Images shape:  torch.Size([16, 1, 1024, 1280])
Images shape:  torch.Size([10, 1, 1024, 1280])
Dataset mean (grayscale): 0.5830
Dataset std (grayscale): 0.2075


## Training - no augmentation

**IMPORTANT:**

- To perform K-Fold cross-validation, set --is_kfold to "True" and specify the number of folds with --k-folds. Example: --is_kfold "True", --k-folds 5

- To perform a single train/val split, set --is_kfold to "False" and specify the validation split ratio with --val-split. Example: --is_kfold "False", --val-split 0.2

- In both cases, to perform also testing, set --test to "True" and specify the test split ratio with --test-split. Example: --test "True", --test-split 0.2

### Define paths and parameters

In [ ]:
data_dir = '/content/mla_project/images'
train_dir = os.path.join(data_dir, 'train')
val_dir   = os.path.join(data_dir, 'val')

# Training params
batch_size = 1
epochs = 10
learning_rate = 1e-3
backbone = 'resnet50'
device = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'
print(f"Using device: {device}")

Using device: cuda


### Launch training

In [ ]:
# k-fold cross validation (with test)

!python /content/mla_project/src/train.py \
    --data-dir "{data_dir}" \
    --batch-size {batch_size} \
    --epochs {10} \
    --lr {learning_rate} \
    --backbone {backbone} \
    --num-workers 2 \
    --is_kfold "True" \
    --k-folds 5 \
    --test "True" \
    --test-split 0.2

### Plot Training & Validation Curves

In [ ]:
# plot for cross-validation
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Read logs
logs = pd.read_csv('/content/kfold_logs.csv')

# Group for epoch and get mean and std
grouped = logs.groupby('epoch').agg({
    'train_loss': ['mean', 'std'],
    'val_loss': ['mean', 'std'],
    'train_acc': ['mean', 'std'],
    'val_acc': ['mean', 'std']
}).reset_index()

# Rename columns
grouped.columns = ['epoch',
                   'train_loss_mean', 'train_loss_std',
                   'val_loss_mean', 'val_loss_std',
                   'train_acc_mean', 'train_acc_std',
                   'val_acc_mean', 'val_acc_std']

# Set style
sns.set(style="white", context="notebook")

# LOSS
plt.figure(figsize=(8, 5))
plt.plot(grouped['epoch'], grouped['train_loss_mean'], label='Train Loss', color='blue')
plt.fill_between(grouped['epoch'],
                 grouped['train_loss_mean'] - grouped['train_loss_std'],
                 grouped['train_loss_mean'] + grouped['train_loss_std'],
                 color='blue', alpha=0.2)

plt.plot(grouped['epoch'], grouped['val_loss_mean'], label='Val Loss', color='orange')
plt.fill_between(grouped['epoch'],
                 grouped['val_loss_mean'] - grouped['val_loss_std'],
                 grouped['val_loss_mean'] + grouped['val_loss_std'],
                 color='orange', alpha=0.2)

plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss (mean ± std)')
plt.legend()
sns.despine()
plt.tight_layout()
plt.show()

# ACCURACY
plt.figure(figsize=(8, 5))
plt.plot(grouped['epoch'], grouped['train_acc_mean'], label='Train Accuracy', color='green')
plt.fill_between(grouped['epoch'],
                 grouped['train_acc_mean'] - grouped['train_acc_std'],
                 grouped['train_acc_mean'] + grouped['train_acc_std'],
                 color='green', alpha=0.2)

plt.plot(grouped['epoch'], grouped['val_acc_mean'], label='Val Accuracy', color='red')
plt.fill_between(grouped['epoch'],
                 grouped['val_acc_mean'] - grouped['val_acc_std'],
                 grouped['val_acc_mean'] + grouped['val_acc_std'],
                 color='red', alpha=0.2)

plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Accuracy (mean ± std)')
plt.legend()
sns.despine()
plt.tight_layout()
plt.show()


## Training - basic augmentations (simple transformations)

In [ ]:
data_dir = '/content/mla_project/images'
train_dir = os.path.join(data_dir, 'train')
val_dir   = os.path.join(data_dir, 'val')

# Training params
batch_size = 2
epochs = 10
learning_rate = 1e-3
backbone = 'resnet50'
device = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'
print(f"Using device: {device}")

Using device: cuda


In [ ]:
# k-fold cross validation (with test) with augmented data

!python /content/mla_project/src/train.py \
    --data-dir "{data_dir}" \
    --batch-size {batch_size} \
    --epochs {10} \
    --lr {learning_rate} \
    --backbone {backbone} \
    --num-workers 2 \
    --is_kfold "True" \
    --k-folds 5 \
    --test "True" \
    --test-split 0.2 \
    --aug "True"

In [ ]:
# plot for cross-validation
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Read logs
logs = pd.read_csv('/content/kfold_logs.csv')

# Group for epoch and get mean and std
grouped = logs.groupby('epoch').agg({
    'train_loss': ['mean', 'std'],
    'val_loss': ['mean', 'std'],
    'train_acc': ['mean', 'std'],
    'val_acc': ['mean', 'std']
}).reset_index()

# Rename columns
grouped.columns = ['epoch',
                   'train_loss_mean', 'train_loss_std',
                   'val_loss_mean', 'val_loss_std',
                   'train_acc_mean', 'train_acc_std',
                   'val_acc_mean', 'val_acc_std']

# Set style
sns.set(style="white", context="notebook")

# LOSS
plt.figure(figsize=(8, 5))
plt.plot(grouped['epoch'], grouped['train_loss_mean'], label='Train Loss', color='blue')
plt.fill_between(grouped['epoch'],
                 grouped['train_loss_mean'] - grouped['train_loss_std'],
                 grouped['train_loss_mean'] + grouped['train_loss_std'],
                 color='blue', alpha=0.2)

plt.plot(grouped['epoch'], grouped['val_loss_mean'], label='Val Loss', color='orange')
plt.fill_between(grouped['epoch'],
                 grouped['val_loss_mean'] - grouped['val_loss_std'],
                 grouped['val_loss_mean'] + grouped['val_loss_std'],
                 color='orange', alpha=0.2)

plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss (mean ± std)')
plt.legend()
sns.despine()
plt.tight_layout()
plt.show()

# ACCURACY
plt.figure(figsize=(8, 5))
plt.plot(grouped['epoch'], grouped['train_acc_mean'], label='Train Accuracy', color='green')
plt.fill_between(grouped['epoch'],
                 grouped['train_acc_mean'] - grouped['train_acc_std'],
                 grouped['train_acc_mean'] + grouped['train_acc_std'],
                 color='green', alpha=0.2)

plt.plot(grouped['epoch'], grouped['val_acc_mean'], label='Val Accuracy', color='red')
plt.fill_between(grouped['epoch'],
                 grouped['val_acc_mean'] - grouped['val_acc_std'],
                 grouped['val_acc_mean'] + grouped['val_acc_std'],
                 color='red', alpha=0.2)

plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Accuracy (mean ± std)')
plt.legend()
sns.despine()
plt.tight_layout()
plt.show()

# Diffusion models

In [2]:
%cd /content/mla_project/external/Diffusion
!git clone https://github.com/huggingface/diffusers
%cd diffusers
!pip install .
!pip install --upgrade diffusers[torch]
!pip install diffusers transformers accelerate scipy safetensors controlnet_aux --upgrade
!pip install -r /content/mla_project/external/Diffusion/diffusers/examples/dreambooth/requirements.txt
!pip install peft==0.15.1
!pip install bitsandbytes

/content/mla_project/external/Diffusion
Cloning into 'diffusers'...
remote: Enumerating objects: 91781, done.
remote: Counting objects: 100% (419/419), done.
remote: Compressing objects: 100% (218/218), done.
remote: Total 91781 (delta 335), reused 202 (delta 200), pack-reused 91362 (from 3)
Receiving objects: 100% (91781/91781), 67.72 MiB | 16.40 MiB/s, done.
Resolving deltas: 100% (67471/67471), done.
/content/mla_project/external/Diffusion/diffusers
Processing /content/mla_project/external/Diffusion/diffusers
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for diffusers: filename=diffusers-0.34.0.dev0-py3-none-any.whl size=3620607 sha256=41b3752cbdf9f6b1e1a496f4e0f9e4f681f7c3094f93f2623cd8336a7dd45d71
  Stored in directory: /tmp/pip-ephem-wheel-cache-m4afgwij/wheels/bf/5f/a5/6ccd758336b2323e9408f3253203e733c2ac9648c12841e5ac
Successfully built diffusers
  Attempting uninstall: diff

## Using Pretrained Model Only and Processing All Folders

In [ ]:
prompt = "a grayscale technical photo of a powder bed surface in a metal additive manufacturing process, containing localized defects such as spattering, holes, incandescence, or uneven lines, high-resolution, flat and metallic texture"
negative_prompt = "artistic, painting, illustration, cartoon, abstract, low quality, blurry, distorted, unrealistic, fantasy, 3d render, color, oversaturated, text, watermark, logo"

!python /content/mla_project/external/Diffusion/generate.py \
    --input_root "/content/mla_project/images/original" \
    --output_root "/content/generated_images_pretrained" \
    --prompt prompt \
    --negative_prompt negative_prompt \
    --strength 0.25 \
    --guidance_scale 4.5 \
    --num_images_per_input 1

2025-05-07 08:41:56.246406: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1746607316.272994   13251 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1746607316.281424   13251 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
Loading pipeline components...: 100% 7/7 [00:14<00:00,  2.13s/it]
100% 10/10 [00:14<00:00,  1.41s/it]
Generated from: Image32.jpg → 1 images
100% 10/10 [00:14<00:00,  1.50s/it]
Generated from: Image14.jpg → 1 images
100% 10/10 [00:15<00:00,  1.53s/it]
Generated from: Image34.jpg → 1 images
100% 10/10 [00:15<00:00,  1.56s/it]
Generated from: Image4.jpg → 1 images
100% 10/10 [00:15<00:00,  1.59s/it]
Generated from: Image45.jpg → 1 imag

## Using Pretrained Model Only and Processing a Single Image

In [ ]:
prompt = "a grayscale technical photo of a powder bed surface in a metal additive manufacturing process, containing localized defects such as spattering, holes, incandescence, or uneven lines, high-resolution, flat and metallic texture"
negative_prompt = "artistic, painting, illustration, cartoon, abstract, low quality, blurry, distorted, unrealistic, fantasy, 3d render, color, oversaturated, text, watermark, logo"

!python /content/mla_project/external/Diffusion/generate.py \
    --path_single_image "/content/mla_project/images/original/Defects/Image2.jpg" \
    --output_root "/content/generated_single_images_pretrained" \
    --prompt prompt \
    --negative_prompt negative_prompt \
    --strength 0.25 \
    --guidance_scale 5.0 \
    --num_images_per_input 1

2025-05-07 08:38:04.682545: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1746607084.719056   12278 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1746607084.725053   12278 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
model_index.json: 100% 541/541 [00:00<00:00, 3.23MB/s]
Fetching 15 files:   0% 0/15 [00:00<?, ?it/s]
model.safetensors:   0% 0.00/1.22G [00:00<?, ?B/s]

model.safetensors:   0% 0.00/492M [00:00<?, ?B/s]
model.safetensors:   3% 31.5M/1.22G [00:00<00:05, 236MB/s]

model.safetensors:   4% 21.0M/492M [00:00<00:03, 148MB/s]


config.json: 100% 4.72k/4.72k [00:00<00:00, 20.5MB/s]



scheduler_config.json: 100% 308/308 [00:00<00:00, 2.39MB/

### Fine tuning Stable Diffusion with LoRA


In [8]:
!accelerate launch /content/mla_project/external/Diffusion/diffusers/examples/dreambooth/train_dreambooth_lora.py \
  --pretrained_model_name_or_path="runwayml/stable-diffusion-v1-5" \
  --instance_data_dir="/content/mla_project/images/original/Defects" \
  --class_data_dir="/content/mla_project/images/original/NoDefects" \
  --output_dir="/content/lora_trained_model" \
  --instance_prompt="a grayscale technical photo of a powder bed surface in a metal additive manufacturing process, containing localized defects such as spattering, holes, incandescence, or uneven lines, high-resolution, flat and metallic texture" \
  --class_prompt="a grayscale technical photo of a uniform powder bed surface in a metal additive manufacturing process, flat and smooth texture, no visible defects, high-resolution" \
  --resolution=512 \
  --train_batch_size=1 \
  --gradient_accumulation_steps=1 \
  --learning_rate=1e-5 \
  --lr_scheduler="constant" \
  --lr_warmup_steps=0 \
  --num_class_images=100 \
  --max_train_steps=400 \
  --checkpointing_steps=100 \
  --seed=42 \
  --mixed_precision="fp16" \
  --use_8bit_adam
  # --lora_r=4 \
  # --lora_alpha=16 \
  # --lora_text_encoder_r=4 \
  # --lora_text_encoder_alpha=16 \


The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_processes` was set to a value of `1`
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
2025-05-07 08:50:45.358266: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1746607845.381286   15499 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1746607845.388595   15499 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-05-07 08:50:45.410609: I tensorflow/core/platform

## Using Fine-Tuned Model and Processing All Folders

In [ ]:
prompt = "a grayscale technical photo of a powder bed surface in a metal additive manufacturing process, containing localized defects such as spattering, holes, incandescence, or uneven lines, high-resolution, flat and metallic texture"
negative_prompt = "artistic, painting, illustration, cartoon, abstract, low quality, blurry, distorted, unrealistic, fantasy, 3d render, color, oversaturated, text, watermark, logo"

#CHECK LORA PATH
!python /content/mla_project/external/Diffusion/generate.py \
    --lora_weights_path "/content/lora_trained_model/pytorch_lora_weights.safetensors" \
    --input_root "/content/mla_project/images/original" \
    --output_root "/content/generated_images_fine_tuned" \
    --prompt prompt \
    --negative_prompt negative_prompt \
    --strength 0.25 \
    --guidance_scale 5.0 \
    --num_images_per_input 1

2025-05-07 08:59:17.084764: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1746608357.104362   17685 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1746608357.110989   17685 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
Loading pipeline components...: 100% 7/7 [00:10<00:00,  1.52s/it]
No LoRA keys associated to CLIPTextModel found with the prefix='text_encoder'. This is safe to ignore if LoRA state dict didn't originally have any CLIPTextModel related params. You can also try specifying `prefix=None` to resolve the warning. Otherwise, open an issue if you think it's unexpected: https://github.com/huggingface/diffusers/issues/new
100% 10/10 [00:14<00

## Using Fine-Tuned Model and Processing a Single Image

In [ ]:
prompt = "a grayscale technical photo of a powder bed surface in a metal additive manufacturing process, containing localized defects such as spattering, holes, incandescence, or uneven lines, high-resolution, flat and metallic texture"
negative_prompt = "artistic, painting, illustration, cartoon, abstract, low quality, blurry, distorted, unrealistic, fantasy, 3d render, color, oversaturated, text, watermark, logo"


!python /content/mla_project/external/Diffusion/generate.py \
    --lora_weights_path "/content/lora_trained_model/pytorch_lora_weights.safetensors" \
    --path_single_image "/content/mla_project/images/original/Defects/Image2.jpg" \
    --output_root "/content/generated_single_images_fine_tuned" \
    --prompt prompt \
    --negative_prompt negative_prompt \
    --strength 0.25 \
    --guidance_scale 5.0 \
    --num_images_per_input 1

2025-05-07 09:02:46.045266: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1746608566.065597   18576 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1746608566.071807   18576 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
Loading pipeline components...: 100% 7/7 [00:14<00:00,  2.12s/it]
No LoRA keys associated to CLIPTextModel found with the prefix='text_encoder'. This is safe to ignore if LoRA state dict didn't originally have any CLIPTextModel related params. You can also try specifying `prefix=None` to resolve the warning. Otherwise, open an issue if you think it's unexpected: https://github.com/huggingface/diffusers/issues/new
100% 10/10 [00:14<00

In [ ]:
import shutil
import os

# 📁 Inserisci qui il percorso della cartella da comprimere
folder_to_zip = "/content/mla_project/images/diffusion"

# 🗜️ Nome dell'archivio di output
zip_filename = "/content/mla_project/images/diffusion.zip"

# ✅ Zippa la cartella
shutil.make_archive(zip_filename.replace(".zip", ""), 'zip', folder_to_zip)
print(f"✅ Cartella compressa in: {zip_filename}")


✅ Cartella compressa in: /content/mla_project/images/diffusion.zip
